# Experiment: fine-tune HiFi-GAN (universal) on the English model's own mels (Colab, L4 GPU)

Why: listening showed the English "robotic" glitch is absent in real recordings and in real mels through the vocoder, but present in our model's output, **including teacher-forced output**. So the acoustic model's mels are over-smoothed, and the vocoder has never seen such mels. Standard remedy (HiFi-GAN paper, sec. 4.4): fine-tune the vocoder on ground-truth-aligned (GTA, teacher-forced) mels paired with the real audio.

Input (upload once to `MyDrive/fs2ra/`): `english_gta.tar`, made locally by
`python scripts/export_gta.py english --ckpt project/English/checkpoints/20000.pth.tar` (experiments branch).

Code: [jik876/hifi-gan](https://github.com/jik876/hifi-gan) @ `4769534` (MIT), with three compatibility edits for current librosa/PyTorch (cell 2). Start point: the official UNIVERSAL_V1 generator + discriminators (`g_02500000`, `do_02500000`), downloaded from the authors' Google Drive into the Colab VM.

Runtime → **L4 GPU**. Training runs in the **background** (survives browser/wifi drops); checkpoints go to `MyDrive/fs2ra/output/English/hifigan_ft/` and a re-run of all cells **resumes**. Use cell 6 to watch progress.

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/fs2ra"
CP = DRIVE + "/output/English/hifigan_ft"          # checkpoints + log (persist)
os.makedirs(CP, exist_ok=True)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU - change runtime type")

In [ ]:
# HiFi-GAN training code, pinned, + compatibility edits (each must apply exactly once)
if not os.path.exists("/content/hifi-gan"):
    !git clone -q https://github.com/jik876/hifi-gan /content/hifi-gan
    !cd /content/hifi-gan && git checkout -q 4769534d45265d52a904b850da5a622601885777

    def edit(path, old, new):
        s = open(path).read()
        assert s.count(old) == 1, (path, old)
        open(path, "w").write(s.replace(old, new))

    md = "/content/hifi-gan/meldataset.py"
    # librosa >= 0.10: keyword-only arguments
    edit(md, "librosa_mel_fn(sampling_rate, n_fft, num_mels, fmin, fmax)",
         "librosa_mel_fn(sr=sampling_rate, n_fft=n_fft, n_mels=num_mels, fmin=fmin, fmax=fmax)")
    # torch >= 2: stft must return complex; take the magnitude as before
    edit(md, "normalized=False, onesided=True)", "normalized=False, onesided=True, return_complex=True)")
    edit(md, "spec = torch.sqrt(spec.pow(2).sum(-1)+(1e-9))",
         "spec = torch.sqrt(torch.view_as_real(spec).pow(2).sum(-1)+(1e-9))")
    # torch >= 2.6 defaults to weights_only=True; the official checkpoints are plain dicts
    edit("/content/hifi-gan/utils.py", "torch.load(filepath, map_location=device)",
         "torch.load(filepath, map_location=device, weights_only=False)")
print("hifi-gan ready")

In [ ]:
# GTA mels + matching audio (from the local export)
if not os.path.exists("/content/gta/training.txt"):
    !cp $DRIVE/english_gta.tar /content/ && tar -xf /content/english_gta.tar -C /content && rm /content/english_gta.tar
!wc -l /content/gta/training.txt /content/gta/validation.txt; ls /content/gta/mels | wc -l

In [ ]:
# Start point: official UNIVERSAL_V1 (same generator as FastSpeech2's bundled generator_universal)
import glob, json, torch
if not glob.glob(CP + "/g_*"):
    !pip -q install gdown
    for fid, name in [("1pAB2kQunkDuv6W5fcJiQ0CY8xcJKB22e", "config.json"),
                      ("1O63eHZR9t1haCdRHQcEgMfMNxiOciSru", "do_02500000"),
                      ("1qpgI41wNXFcH-iKq1Y42JlBC9j0je8PW", "g_02500000")]:
        !gdown -q $fid -O $CP/$name
if not os.path.exists(CP + "/plan.json"):
    base = torch.load(CP + "/do_02500000", map_location="cpu", weights_only=False)
    EPOCHS = 50          # ~213 steps/epoch (3,417 utts, batch 16) -> ~10.7k fine-tuning steps
    json.dump({"base_steps": base["steps"], "base_epoch": base["epoch"], "training_epochs": base["epoch"] + EPOCHS},
              open(CP + "/plan.json", "w"))
plan = json.load(open(CP + "/plan.json")); print(plan)
print(json.load(open(CP + "/config.json")))

In [ ]:
# Fine-tuning in the background (own process session): a browser/wifi drop does not stop it.
# Re-running this cell while it runs does nothing; after a runtime reset it resumes from the newest g_/do_ in CP.
LOG = CP + "/train.log"
if subprocess.run("pgrep -f '[t]rain.py --config'", shell=True).returncode == 0:
    print("already running - use the next cell")
else:
    cmd = ["python", "train.py", "--config", CP + "/config.json", "--fine_tuning", "True",
           "--input_wavs_dir", "/content/gta/wavs", "--input_mels_dir", "/content/gta/mels",
           "--input_training_file", "/content/gta/training.txt",
           "--input_validation_file", "/content/gta/validation.txt",
           "--checkpoint_path", CP, "--training_epochs", str(plan["training_epochs"]),
           "--checkpoint_interval", "1000", "--validation_interval", "1000", "--stdout_interval", "100"]
    subprocess.Popen(cmd, cwd="/content/hifi-gan", stdout=open(LOG, "a"), stderr=subprocess.STDOUT,
                     start_new_session=True)
    # keep only the newest 2 checkpoint pairs on Drive (each do_ holds both optimisers, ~1 GB)
    prune = ("while true; do for p in g_ do_; do ls -1 \"%s\"/$p* 2>/dev/null | sort | head -n -2 "
             "| xargs -r -d '\n' rm -f; done; sleep 120; done") % CP
    subprocess.Popen(["bash", "-c", prune], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    print("started; log:", LOG)

In [ ]:
# Progress: re-run any time. "Mel-Spec. Error" = train L1; validation L1 from TensorBoard logs.
!tail -n 4 $CP/train.log; pgrep -f '[t]rain.py --config' > /dev/null && echo "== running ==" || echo "== not running =="
!ls -1 $CP | grep -E '^(g|do)_'
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
ev = EventAccumulator(CP + "/logs"); ev.Reload()
if "validation/mel_spec_error" in ev.Tags()["scalars"]:
    for e in ev.Scalars("validation/mel_spec_error")[-8:]: print("step %d  val mel L1 %.4f" % (e.step, e.value))

In [ ]:
# Stop when satisfied (or when validation L1 stops improving); then download the newest g_XXXXXXXX
# (generator only, ~55 MB) from MyDrive/fs2ra/output/English/hifigan_ft/.
!pkill -f '[t]rain.py --config'; pkill -f '[s]leep 120'; pkill -f '[h]ead -n -2'; echo stopped